# SiteGuard-CP: Shift-aware conformal triage for psychiatric connectomes at unseen sites

**Workshop:** International Workshop on AI in Mental Health, NIT Goa (29-30 Dec 2026)

**Question this notebook answers:** a connectome classifier trained on some hospitals is deployed at a *new* hospital.
Does its conformal "90% guarantee" still hold there, and can we tell **before collecting any labels** at that hospital?

**What runs here (CPU only, ~10-25 min on free Colab after the one-time download):**
1. ABIDE-I (871 QC-passed subjects, 20 site cohorts) ROI time series from the Preprocessed Connectomes Project, AAL atlas (116 ROIs -> 6,670 edges).
2. Leakage-free leave-one-site-out (LOSO): tangent-space connectivity and the classifier are fit on training sites only.
3. Five conformal predictors compared at every held-out site:
   * `marginal` split conformal (standard)
   * `mondrian` class-conditional conformal (current best practice, Rokham et al. 2026)
   * `wcp` weighted conformal under covariate shift (Tibshirani et al. 2019)
   * `wm_cov` **weighted Mondrian** conformal (proposed, covariate-shift weights inside each class)
   * `wm_hyb` **weighted Mondrian with hybrid shift weights** (proposed, corrects covariate shift and site prevalence shift together)
   All shift weights use only **unlabeled** scans from the new site, with **ESS-constrained tempering** so weights never collapse.
4. **Site Readiness Score (SRS):** a label-free estimate of how much coverage the standard method will lose at a new site, tested against the true loss.
5. **Conformally gated biomarkers:** are edge attributions from confidently classified subjects more stable across sites than attributions from everyone?

Set `USE_SYNTHETIC=True` to smoke-test the whole pipeline in about a minute without downloading anything.

In [ ]:
# ---------- 0. Setup ----------
import sys, subprocess, os, time, json, warnings
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nilearn>=0.10'], check=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, wilcoxon, beta
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
warnings.filterwarnings('ignore')
from IPython.display import display

CFG = dict(
    USE_SYNTHETIC=False,        # True = quick smoke test, no download
    DERIVATIVE='rois_aal',      # AAL 116 ROIs -> 6,670 edges. Try 'rois_cc200' later (19,900 edges, slower)
    CONN_KIND='tangent',        # 'tangent' (fit per fold on training sites) or 'correlation' (Fisher z)
    HARMONIZE='none',           # 'none' or 'site_z' (label-free per-site z-scoring, an ablation)
    ALPHA=0.10,                 # target miscoverage -> 90% prediction sets
    CAL_FRAC=0.30,              # share of training-site subjects held out for calibration
    N_REPEATS=5,                # random calibration splits per held-out site
    MIN_TEST_SITE_N=20,         # sites smaller than this train other folds but are not scored
    DOMAIN_SPACE='pca',         # 'pca' (PCA of connectome) or 'score' (classifier logit only)
    PCA_DOMAIN=20,              # dimensions for the domain classifier
    ESS_MIN=0.20,               # tempering keeps effective sample size >= 20% of calibration set
    W_CLIP=20.0,                # hard clip on density ratios
    C_LR=0.05,                  # inverse L2 strength of the diagnosis classifier
    TOPK=100,                   # edges per fold for stability analysis
    GATE_PRED_GAP=0.03,         # SRS gate: flag site if predicted coverage loss > 3 points ...
    GATE_ESS=0.10,              # ... or untempered ESS fraction < 10%
    FAIL_GAP=0.05,              # a site "fails" if observed Mondrian coverage < 1-alpha-0.05
    SEED=42,
    CACHE_FILE='siteguard_cache.pkl',  # per-site probabilities + domain features, saved inside OUT_DIR
    DATA_DIR='/content/abide_data' if IN_COLAB else './abide_data',
    OUT_DIR='./siteguard_results',
)
os.makedirs(CFG['OUT_DIR'], exist_ok=True)
rng_global = np.random.default_rng(CFG['SEED'])
print(json.dumps(CFG, indent=1))

## 1. Data
`fetch_abide_pcp` downloads one small text file per subject (about 150 MB in total for AAL). It is cached in `DATA_DIR`,
so mount Google Drive and point `DATA_DIR` there if you want to skip the download next session.

In [ ]:
# ---------- 1. Load ABIDE-I (or synthetic) ----------
def _to_str(v):
    return v.decode() if isinstance(v, (bytes, np.bytes_)) else str(v)

def load_abide(cfg):
    from nilearn.datasets import fetch_abide_pcp
    d = fetch_abide_pcp(data_dir=cfg['DATA_DIR'], pipeline='cpac', band_pass_filtering=True,
                        global_signal_regression=False, derivatives=[cfg['DERIVATIVE']],
                        quality_checked=True, verbose=1)
    ts = [np.asarray(t, dtype=float) for t in d[cfg['DERIVATIVE']]]
    ph = pd.DataFrame(d['phenotypic'])
    y = (ph['DX_GROUP'].astype(int).values == 1).astype(int)            # 1 = ASD, 0 = typical control
    site = np.array([_to_str(s) for s in ph['SITE_ID'].values])
    keep = np.array([t.ndim == 2 and t.shape[0] >= 50 and np.isfinite(t).all() for t in ts])
    ts = [t for t, k in zip(ts, keep) if k]
    return ts, y[keep], site[keep], ph.loc[keep].reset_index(drop=True)

def make_synthetic(n_sites=10, R=30, seed=0):
    """Toy multi-site data: class effect in a fixed network block, plus site-specific
    low-rank connectivity distortions, site-specific noise and site-specific prevalence."""
    rng = np.random.default_rng(seed)
    A = rng.normal(size=(R, R // 3)); C0 = A @ A.T + R * 0.3 * np.eye(R)
    dsd = np.sqrt(np.diag(C0)); C0 = C0 / np.outer(dsd, dsd)
    u = np.zeros(R); u[:6] = 1.0; u[6:12] = -0.7; u /= np.linalg.norm(u)
    ts, y, site = [], [], []
    for s in range(n_sites):
        n = rng.integers(25, 70); prev = rng.uniform(0.3, 0.7)
        T = int(rng.integers(100, 220)); v = rng.normal(size=R); v /= np.linalg.norm(v)
        site_strength = rng.uniform(0.2, 1.5); noise = rng.uniform(0.3, 1.0)
        for _ in range(n):
            lab = int(rng.random() < prev)
            C = C0 + 0.9 * lab * np.outer(u, u) + site_strength * np.outer(v, v) + noise * np.eye(R)
            L = np.linalg.cholesky(C)
            ts.append(rng.normal(size=(T, R)) @ L.T); y.append(lab); site.append(f'SITE_{s:02d}')
    return ts, np.array(y), np.array(site), None

t0 = time.time()
if CFG['USE_SYNTHETIC']:
    TS, Y, SITE, PHENO = make_synthetic(seed=CFG['SEED'])
else:
    TS, Y, SITE, PHENO = load_abide(CFG)
N_ROI = TS[0].shape[1]
print(f'{len(TS)} subjects, {N_ROI} ROIs, {len(np.unique(SITE))} sites, ASD share = {Y.mean():.2f}  ({time.time()-t0:.0f}s)')
site_tab = pd.crosstab(pd.Series(SITE, name='site'), pd.Series(Y, name='label')).rename(columns={0: 'TC', 1: 'ASD'})
site_tab['n'] = site_tab.sum(1); site_tab['ASD_share'] = (site_tab['ASD'] / site_tab['n']).round(2)
display(site_tab.sort_values('n', ascending=False))

# ROI labels for the biomarker table
ROI_LABELS = [f'ROI_{i}' for i in range(N_ROI)]
if not CFG['USE_SYNTHETIC'] and CFG['DERIVATIVE'] == 'rois_aal':
    try:
        from nilearn.datasets import fetch_atlas_aal
        _lab = list(fetch_atlas_aal(data_dir=CFG['DATA_DIR']).labels)
        _lab = [l for l in _lab if str(l).lower() != 'background']
        if len(_lab) == N_ROI:
            ROI_LABELS = [str(l) for l in _lab]
    except Exception as e:
        print('AAL labels unavailable, using indices:', e)

## 2. Connectivity features and the conformal toolbox
All shift weights are computed from **unlabeled** target-site scans. Labels at the held-out site are used only for scoring.

In [ ]:
# ---------- 2a. Connectivity ----------
from nilearn.connectome import ConnectivityMeasure
TRIL = np.tril_indices(N_ROI, k=-1)          # same edge order as nilearn's vectorize(discard_diagonal=True)

def corr_features(ts_list):
    out = np.empty((len(ts_list), len(TRIL[0])))
    for i, t in enumerate(ts_list):
        c = np.corrcoef(t, rowvar=False); c = np.nan_to_num(c)
        out[i] = np.arctanh(np.clip(c[TRIL], -0.999, 0.999))
    return out

def fold_features(train_idx, test_idx):
    if CFG['CONN_KIND'] == 'correlation':
        return CORR_ALL[train_idx], CORR_ALL[test_idx]
    cm = ConnectivityMeasure(kind='tangent', vectorize=True, discard_diagonal=True)
    Xtr = cm.fit_transform([TS[i] for i in train_idx])      # reference mean from training sites only
    Xte = cm.transform([TS[i] for i in test_idx])
    return np.nan_to_num(Xtr), np.nan_to_num(Xte)

def site_z(X, sites):
    """Label-free per-site z-scoring (uses each site's own unlabeled statistics)."""
    Xz = np.empty_like(X)
    for s in np.unique(sites):
        m = sites == s; mu = X[m].mean(0); sd = X[m].std(0) + 1e-6
        Xz[m] = (X[m] - mu) / sd
    return Xz

CORR_ALL = corr_features(TS) if CFG['CONN_KIND'] == 'correlation' else None

In [ ]:
# ---------- 2b. Conformal toolbox (no data needed; run this before the replay cells in a fresh session) ----------
def lac(P):
    """LAC nonconformity: s(x, y) = 1 - p_y(x). Returns an (n, K) score matrix."""
    return 1.0 - P

def threshold(cal_s, alpha, n_test, cal_w=None, test_w=None):
    """Split-conformal threshold per test point. Unweighted: ceil((n+1)(1-a))-th score.
    Weighted (Tibshirani et al. 2019): smallest s with sum_{s_i<=s} w_i >= (1-a)(sum w + w_test)."""
    if len(cal_s) == 0:
        return np.full(n_test, np.inf)
    order = np.argsort(cal_s); s = cal_s[order]
    if cal_w is None:
        k = int(np.ceil((len(s) + 1) * (1 - alpha)))
        return np.full(n_test, np.inf if k > len(s) else s[k - 1])
    cw = np.cumsum(cal_w[order]); target = (1 - alpha) * (cw[-1] + test_w)
    idx = np.searchsorted(cw, target, side='left')
    return np.where(idx < len(s), s[np.minimum(idx, len(s) - 1)], np.inf)

def ess_frac(w):
    return (w.sum() ** 2) / (np.square(w).sum() * len(w))

def temper(r_cal, r_tgt, ess_min):
    """Largest gamma in [0,1] with ESS(r_cal**gamma) >= ess_min. gamma=0 -> plain conformal."""
    for g in np.linspace(1, 0, 21):
        if ess_frac(r_cal ** g) >= ess_min:
            return g, r_cal ** g, r_tgt ** g
    return 0.0, np.ones_like(r_cal), np.ones_like(r_tgt)

def density_ratio(Z_cal, Z_tgt, seed, clip):
    """Cross-fitted domain classifier calibration-vs-target. Returns r(x)=p_T(x)/p_S(x) up to a constant, and AUC."""
    Xd = np.vstack([Z_cal, Z_tgt]); d = np.r_[np.zeros(len(Z_cal)), np.ones(len(Z_tgt))]
    n_splits = int(min(5, len(Z_tgt), len(Z_cal)))
    clf = LogisticRegression(C=0.1, max_iter=3000)
    p = cross_val_predict(clf, StandardScaler().fit_transform(Xd), d,
                          cv=StratifiedKFold(n_splits, shuffle=True, random_state=seed), method='predict_proba')[:, 1]
    auc = roc_auc_score(d, p); p = np.clip(p, 1e-4, 1 - 1e-4)
    r = np.clip(p / (1 - p), 1 / clip, clip); r = r / r[:len(Z_cal)].mean()
    return r[:len(Z_cal)], r[len(Z_cal):], auc

def em_prior(P_tgt, pi_src, iters=100):
    """Saerens-Latinne-Decaestecker EM: target class prior from unlabeled target posteriors."""
    pi = pi_src.copy()
    for _ in range(iters):
        q = P_tgt * (pi / pi_src); q /= q.sum(1, keepdims=True)
        new = np.clip(q.mean(0), 0.05, 0.95); new /= new.sum()
        if np.abs(new - pi).max() < 1e-6: break
        pi = new
    return pi

def hybrid_factor(P, pi_src, pi_tgt):
    """1/Z(x), Z(x)=sum_j p_S(j|x) pi_T(j)/pi_S(j). Turns covariate-shift weights into class-conditional
    weights p_T(x|k)/p_S(x|k) up to a per-class constant, exact under pure label shift."""
    return 1.0 / (P * (pi_tgt / pi_src)).sum(1)

def sets_from(S_test, thr):          # thr: (n_test, K)
    return S_test <= thr

def evaluate(sets, y):
    n = len(y); size = sets.sum(1); cover = sets[np.arange(n), y]
    single = size == 1; pred = sets.argmax(1)
    out = dict(coverage=cover.mean(), set_size=size.mean(), singleton=single.mean(), empty_set=(size == 0).mean(),
               refer=(size >= 2).mean(),
               single_acc=(pred[single] == y[single]).mean() if single.any() else np.nan)
    for k in (0, 1):
        out[f'cov_class{k}'] = cover[y == k].mean() if (y == k).any() else np.nan
    return out

## 3. Leave-one-site-out experiment
For each held-out site: fit connectivity + classifier on the other sites, split their subjects into proper-train / calibration
(stratified by site x label), then build all five predictors for the held-out site and score them.

In [ ]:
# ---------- 3. LOSO loop ----------
ALPHA = CFG['ALPHA']
sites = np.unique(SITE)
scored_sites = [s for s in sites if (SITE == s).sum() >= CFG['MIN_TEST_SITE_N'] and len(np.unique(Y[SITE == s])) == 2]
rows, srs_rows, base_rows, bio, CACHE = [], [], [], {}, []
t0 = time.time()
for si, s_out in enumerate(scored_sites):
    te = np.where(SITE == s_out)[0]; pool = np.where(SITE != s_out)[0]
    Xpool, Xte = fold_features(pool, te)
    if CFG['HARMONIZE'] == 'site_z':
        Xpool = site_z(Xpool, SITE[pool]); Xte = site_z(Xte, SITE[te])
    yte = Y[te]
    for rep in range(CFG['N_REPEATS']):
        seed = CFG['SEED'] + 1000 * si + rep
        strat = np.array([f'{a}_{b}' for a, b in zip(SITE[pool], Y[pool])])
        _, cnt = np.unique(strat, return_counts=True)
        strat_ok = strat if cnt.min() >= 2 else Y[pool]
        itr, ical = train_test_split(np.arange(len(pool)), test_size=CFG['CAL_FRAC'], stratify=strat_ok, random_state=seed)
        scaler = StandardScaler().fit(Xpool[itr])
        Xtr_s, Xcal_s, Xte_s = scaler.transform(Xpool[itr]), scaler.transform(Xpool[ical]), scaler.transform(Xte)
        ytr, ycal = Y[pool][itr], Y[pool][ical]
        clf = LogisticRegression(C=CFG['C_LR'], max_iter=5000).fit(Xtr_s, ytr)
        Pcal, Pte = clf.predict_proba(Xcal_s), clf.predict_proba(Xte_s)
        Scal, Ste = lac(Pcal), lac(Pte)
        scal_true = Scal[np.arange(len(ycal)), ycal]
        nT = len(te)

        # --- label-free shift weights (target labels never touched) ---
        if CFG['DOMAIN_SPACE'] == 'pca':
            pca = PCA(n_components=min(CFG['PCA_DOMAIN'], Xtr_s.shape[1], len(itr) - 1), random_state=seed).fit(Xtr_s)
            Zc, Zt = pca.transform(Xcal_s), pca.transform(Xte_s)
        else:
            lg = lambda P: np.log(P[:, 1:2] + 1e-9) - np.log(P[:, 0:1] + 1e-9)
            Zc, Zt = lg(Pcal), lg(Pte)
        r_cal, r_te, dom_auc = density_ratio(Zc, Zt, seed, CFG['W_CLIP'])
        ess_raw = ess_frac(r_cal)
        g_cov, wc_cov, wt_cov = temper(r_cal, r_te, CFG['ESS_MIN'])
        pi_src = np.bincount(ycal, minlength=2) / len(ycal)
        pi_tgt = em_prior(Pte, pi_src)
        h_cal, h_te = r_cal * hybrid_factor(Pcal, pi_src, pi_tgt), r_te * hybrid_factor(Pte, pi_src, pi_tgt)
        g_hyb, wc_hyb, wt_hyb = temper(h_cal, h_te, CFG['ESS_MIN'])

        # --- thresholds (n_test, K) per method ---
        thr = {}
        thr['marginal'] = np.repeat(threshold(scal_true, ALPHA, nT)[:, None], 2, 1)
        thr['wcp'] = np.repeat(threshold(scal_true, ALPHA, nT, wc_cov, wt_cov)[:, None], 2, 1)
        for name in ('mondrian', 'wm_cov', 'wm_hyb'):
            thr[name] = np.zeros((nT, 2))
        for k in (0, 1):
            m = ycal == k
            thr['mondrian'][:, k] = threshold(scal_true[m], ALPHA, nT)
            thr['wm_cov'][:, k] = threshold(scal_true[m], ALPHA, nT, wc_cov[m], wt_cov)
            thr['wm_hyb'][:, k] = threshold(scal_true[m], ALPHA, nT, wc_hyb[m], wt_hyb)
        sets_by = {}
        for name, T_ in thr.items():
            sets_by[name] = sets_from(Ste, T_)
            rows.append(dict(site=s_out, n=nT, asd_share=yte.mean(), rep=rep, method=name, **evaluate(sets_by[name], yte)))

        # --- Site Readiness Score: label-free predicted coverage of plain Mondrian at this site ---
        est_cov = 0.0
        for k in (0, 1):
            m = ycal == k
            q_k = thr['mondrian'][0, k]
            est_cov += pi_tgt[k] * np.sum(h_cal[m] * (scal_true[m] <= q_k)) / np.sum(h_cal[m])
        srs_rows.append(dict(site=s_out, rep=rep, dom_auc=dom_auc, ess_raw=ess_raw, gamma_cov=g_cov, gamma_hyb=g_hyb,
                             pi_tgt_est=pi_tgt[1], pi_tgt_true=yte.mean(), pred_cov_mondrian=est_cov,
                             pred_gap=(1 - ALPHA) - est_cov))
        # --- cache what the replay cells need (no features, no subject IDs) ---
        CACHE.append(dict(site=s_out, rep=rep, seed=seed, ycal=ycal, yte=yte,
                          Pcal=Pcal.astype(np.float64), Pte=Pte.astype(np.float64),
                          Zc=np.asarray(Zc, np.float32), Zt=np.asarray(Zt, np.float32)))
        # --- base classifier ---
        base_rows.append(dict(site=s_out, rep=rep, acc=accuracy_score(yte, Pte.argmax(1)),
                              bacc=balanced_accuracy_score(yte, Pte.argmax(1)),
                              auc=roc_auc_score(yte, Pte[:, 1])))
        # --- biomarker material (first repeat only) ---
        if rep == 0:
            coef = clf.coef_.ravel(); A = Xte_s * coef          # exact additive attribution for a linear model
            conf = sets_by['wm_hyb'].sum(1) == 1
            bio[s_out] = dict(coef=np.abs(coef), all=np.abs(A).mean(0),
                              conf=np.abs(A[conf]).mean(0) if conf.sum() >= 3 else np.abs(A).mean(0),
                              n_conf=int(conf.sum()), n=nT)
    print(f'[{si+1:2d}/{len(scored_sites)}] {s_out:12s} n={len(te):3d}  dom_AUC={dom_auc:.2f}  '
          f'gamma_hyb={g_hyb:.2f}  ({time.time()-t0:.0f}s)')

R = pd.DataFrame(rows); SRS = pd.DataFrame(srs_rows); BASE = pd.DataFrame(base_rows)
R.to_csv(f"{CFG['OUT_DIR']}/per_site_results.csv", index=False)
SRS.to_csv(f"{CFG['OUT_DIR']}/site_readiness.csv", index=False)
BASE.to_csv(f"{CFG['OUT_DIR']}/base_classifier.csv", index=False)
import pickle
with open(f"{CFG['OUT_DIR']}/{CFG['CACHE_FILE']}", 'wb') as f:
    pickle.dump(dict(cache=CACHE, cfg=CFG), f)
print('done in', round(time.time() - t0), 's')

## 3b. Replay: recompute every conformal method from the cache in seconds
The LOSO loop saves each site's calibration/test probabilities and domain features. Everything below the classifier
(any alpha, ESS floor, weight clip, domain space, and the matched-set-size comparison) is recomputed here without refitting.
In a fresh session: run cell 0 (setup) and cell 2b (toolbox), then these cells. No ABIDE download needed.

In [ ]:
# ---------- 3b. Replay engine ----------
import pickle
if 'CACHE' not in globals() or len(CACHE) == 0:
    with open(f"{CFG['OUT_DIR']}/{CFG['CACHE_FILE']}", 'rb') as f:
        CACHE = pickle.load(f)['cache']
print(len(CACHE), 'cached site x split entries')

def replay(e, alpha=None, ess_min=None, w_clip=None, domain='pca', methods=('marginal', 'mondrian', 'wcp', 'wm_cov', 'wm_hyb')):
    alpha = CFG['ALPHA'] if alpha is None else alpha
    ess_min = CFG['ESS_MIN'] if ess_min is None else ess_min
    w_clip = CFG['W_CLIP'] if w_clip is None else w_clip
    ycal, yte, Pcal, Pte = e['ycal'], e['yte'], e['Pcal'], e['Pte']
    Scal, Ste = lac(Pcal), lac(Pte); st = Scal[np.arange(len(ycal)), ycal]; nT = len(yte)
    if domain == 'pca':
        Zc, Zt = e['Zc'].astype(float), e['Zt'].astype(float)
    else:
        lg = lambda P: np.log(P[:, 1:2] + 1e-9) - np.log(P[:, 0:1] + 1e-9)
        Zc, Zt = lg(Pcal), lg(Pte)
    r_cal, r_te, _ = density_ratio(Zc, Zt, e['seed'], w_clip)
    _, wc, wt = temper(r_cal, r_te, ess_min)
    pi_src = np.bincount(ycal, minlength=2) / len(ycal); pi_tgt = em_prior(Pte, pi_src)
    h_cal, h_te = r_cal * hybrid_factor(Pcal, pi_src, pi_tgt), r_te * hybrid_factor(Pte, pi_src, pi_tgt)
    _, hc, ht = temper(h_cal, h_te, ess_min)
    out = []
    for name in methods:
        if name == 'marginal':
            T_ = np.repeat(threshold(st, alpha, nT)[:, None], 2, 1)
        elif name == 'wcp':
            T_ = np.repeat(threshold(st, alpha, nT, wc, wt)[:, None], 2, 1)
        else:
            T_ = np.zeros((nT, 2))
            for k in (0, 1):
                m = ycal == k
                if name == 'mondrian': T_[:, k] = threshold(st[m], alpha, nT)
                elif name == 'wm_cov': T_[:, k] = threshold(st[m], alpha, nT, wc[m], wt)
                else: T_[:, k] = threshold(st[m], alpha, nT, hc[m], ht)
        out.append(dict(site=e['site'], rep=e['rep'], n=nT, method=name, alpha=alpha, **evaluate(sets_from(Ste, T_), yte)))
    return out

def replay_all(**kw):
    df = pd.DataFrame([r for e in CACHE for r in replay(e, **kw)])
    return df.groupby(['method', 'site']).mean(numeric_only=True).reset_index()

def headline(sm, target):
    g = sm.groupby('method')
    return pd.DataFrame(dict(
        mean_cov=g.coverage.mean(), worst_site_cov=g.coverage.min(),
        n_fail_sites=g.coverage.apply(lambda c: int((c < target - CFG['FAIL_GAP']).sum())),
        mean_shortfall=g.coverage.apply(lambda c: np.clip(target - c, 0, None).mean()),
        set_size=g.set_size.mean(), singleton=g.singleton.mean(), single_acc=g.single_acc.mean())).round(3)

# sanity check: replay at the default settings must reproduce the LOSO loop exactly
if 'R' in globals() and len(R):
    chk = replay_all()
    ref = R.groupby(['method', 'site']).mean(numeric_only=True).reset_index()
    mrg = chk.merge(ref, on=['method', 'site'], suffixes=('', '_loop'))
    print('max |coverage difference| replay vs loop:', float(np.abs(mrg.coverage - mrg.coverage_loop).max()))

In [ ]:
# ---------- 3c. The key test: Mondrian at matched set size ----------
# A reviewer will ask whether wm_cov only wins by making bigger sets. Here Mondrian gets a stricter alpha, chosen
# ON THE TEST SITES so that its mean set size matches wm_cov's. That tuning favours the baseline, so it is a conservative test.
target = 1 - CFG['ALPHA']
base_sm = replay_all(methods=('wm_cov',))
size_star = base_sm.set_size.mean()
grid = np.round(np.arange(0.010, CFG['ALPHA'] + 1e-9, 0.005), 3)
sizes = {a: replay_all(alpha=a, methods=('mondrian',)).set_size.mean() for a in grid}
a_match = min(grid, key=lambda a: (abs(sizes[a] - size_star), -a))
mm = replay_all(alpha=a_match, methods=('mondrian',)); mm['method'] = f'mondrian@a={a_match}'
MATCHED = headline(pd.concat([base_sm, mm, replay_all(methods=('mondrian',))]), target)
print(f'wm_cov mean set size at alpha={CFG["ALPHA"]}: {size_star:.3f}; Mondrian matched at alpha={a_match} (size {sizes[a_match]:.3f})')
display(MATCHED)
pv = pd.concat([base_sm, mm]).pivot(index='site', columns='method', values='coverage')
d_w = np.clip(target - pv['wm_cov'], 0, None); d_m = np.clip(target - pv[mm.method.iloc[0]], 0, None)
try:
    p = wilcoxon(d_w, d_m, alternative='less', zero_method='zsplit').pvalue
except Exception:
    p = np.nan
print(f'shortfall wm_cov {d_w.mean():.3f} vs matched Mondrian {d_m.mean():.3f}; one-sided Wilcoxon p = {p:.3f}')
MATCHED.to_csv(f"{CFG['OUT_DIR']}/matched_set_size.csv")

In [ ]:
# ---------- 3d. Sweeps that need no refitting (ablations A3, A4, A5) ----------
sweep = []
for a in (0.05, 0.10, 0.20):
    h = headline(replay_all(alpha=a), 1 - a); h['alpha'] = a; h['ess_min'] = CFG['ESS_MIN']; h['domain'] = 'pca'; sweep.append(h)
for em in (0.05, 0.10, 0.40):
    h = headline(replay_all(ess_min=em, methods=('wcp', 'wm_cov', 'wm_hyb')), target); h['alpha'] = CFG['ALPHA']; h['ess_min'] = em; h['domain'] = 'pca'; sweep.append(h)
h = headline(replay_all(domain='score', methods=('wcp', 'wm_cov', 'wm_hyb')), target); h['alpha'] = CFG['ALPHA']; h['ess_min'] = CFG['ESS_MIN']; h['domain'] = 'score'; sweep.append(h)
SWEEP = pd.concat(sweep).reset_index().set_index(['alpha', 'ess_min', 'domain', 'method'])
SWEEP.to_csv(f"{CFG['OUT_DIR']}/sweeps.csv")
display(SWEEP)

In [ ]:
# ---------- 3e. Diagnose the hybrid weights and the readiness score (replay only, seconds) ----------
# Run 1 showed the hybrid method fails exactly where the EM estimate of the site's ASD share is wrong.
# Here we swap the prior estimator: 'em' (as before), 'bbse' (soft black-box shift estimation, Lipton et al. 2018),
# 'source' (no label-shift correction), and 'oracle' (TRUE target share: a diagnostic ceiling only, never a method).
def est_prior(kind, Pcal, ycal, Pte, pi_src):
    if kind == 'em':
        return em_prior(Pte, pi_src)
    if kind == 'source':
        return pi_src.copy()
    if kind == 'bbse':
        C = np.stack([Pcal[ycal == j].mean(0) * pi_src[j] for j in (0, 1)], axis=1)   # C[i,j] = E_S[p(i|x) 1{y=j}]
        mu = Pte.mean(0)
        try:
            w = np.linalg.solve(C, mu)
        except np.linalg.LinAlgError:
            return pi_src.copy()
        pi = np.clip(w * pi_src, 0.05, 0.95); return pi / pi.sum()
    raise ValueError(kind)

def replay_diag(e, prior='em', alpha=None):
    alpha = CFG['ALPHA'] if alpha is None else alpha
    ycal, yte, Pcal, Pte = e['ycal'], e['yte'], e['Pcal'], e['Pte']
    Scal, Ste = lac(Pcal), lac(Pte); st = Scal[np.arange(len(ycal)), ycal]; nT = len(yte)
    r_cal, r_te, _ = density_ratio(e['Zc'].astype(float), e['Zt'].astype(float), e['seed'], CFG['W_CLIP'])
    pi_src = np.bincount(ycal, minlength=2) / len(ycal)
    if prior == 'oracle':
        pi_tgt = np.array([1 - yte.mean(), yte.mean()])          # uses target labels: diagnostic only
    else:
        pi_tgt = est_prior(prior, Pcal, ycal, Pte, pi_src)
    h_cal, h_te = r_cal * hybrid_factor(Pcal, pi_src, pi_tgt), r_te * hybrid_factor(Pte, pi_src, pi_tgt)
    _, hc, ht = temper(h_cal, h_te, CFG['ESS_MIN'])
    _, wc, _ = temper(r_cal, r_te, CFG['ESS_MIN'])
    T_ = np.zeros((nT, 2)); qm = np.zeros(2)
    for k in (0, 1):
        m = ycal == k
        T_[:, k] = threshold(st[m], alpha, nT, hc[m], ht)
        qm[k] = threshold(st[m], alpha, 1)[0]
    ok = st <= qm[ycal]                                          # calibration points Mondrian would cover
    srs_hyb = sum(pi_tgt[k] * np.sum(h_cal[ycal == k] * ok[ycal == k]) / np.sum(h_cal[ycal == k]) for k in (0, 1))
    srs_cov = np.sum(wc * ok) / np.sum(wc)                       # covariate-only readiness score, no prior needed
    return dict(site=e['site'], rep=e['rep'], prior=prior, prior_err=abs(pi_tgt[1] - yte.mean()),
                srs_hyb=srs_hyb, srs_cov=srs_cov, **evaluate(sets_from(Ste, T_), yte))

target = 1 - CFG['ALPHA']
D = pd.DataFrame([replay_diag(e, pr) for e in CACHE for pr in ('em', 'bbse', 'source', 'oracle')])
Dm = D.groupby(['prior', 'site']).mean(numeric_only=True).reset_index()
mond = replay_all(methods=('mondrian',)).set_index('site').coverage
cov_ref = replay_all(methods=('wm_cov',)).set_index('site').coverage
rows_ = []
for pr, g in Dm.groupby('prior'):
    g = g.set_index('site'); obs_gap = target - mond.loc[g.index]
    rows_.append(dict(prior=pr, prior_MAE=g.prior_err.mean(), worst_site_cov=g.coverage.min(),
                      n_fail_sites=int((g.coverage < target - CFG['FAIL_GAP']).sum()),
                      mean_shortfall=np.clip(target - g.coverage, 0, None).mean(), set_size=g.set_size.mean(),
                      rho_srs_hyb=spearmanr(target - g.srs_hyb, obs_gap)[0],
                      rho_srs_cov=spearmanr(target - g.srs_cov, obs_gap)[0],
                      rho_err_vs_loss=spearmanr(g.prior_err, (g.coverage - cov_ref.loc[g.index]))[0]))
DIAG = pd.DataFrame(rows_).set_index('prior').loc[['em', 'bbse', 'source', 'oracle']].round(3)
print('wm_hyb under different prior estimators (oracle = uses target labels, diagnostic ceiling only)')
display(DIAG)
print('rho_srs_*: Spearman between predicted and observed Mondrian coverage loss across sites.')
print('rho_err_vs_loss: Spearman between prior error and (wm_hyb - wm_cov) coverage; strongly negative = prior error drives the failure.')
DIAG.to_csv(f"{CFG['OUT_DIR']}/hybrid_prior_diagnosis.csv")

In [ ]:
# ---------- 3f. Efficiency frontier: coverage vs set size over an alpha grid (replay, about 1-3 min) ----------
# The fair comparison: at the SAME mean set size, which method has better worst-site coverage and less shortfall?
# A method only helps if its curve lies above Mondrian's.
def weights_for(e, domain):
    if domain == 'pca':
        Zc, Zt = e['Zc'].astype(float), e['Zt'].astype(float)
    else:
        lg = lambda P: np.log(P[:, 1:2] + 1e-9) - np.log(P[:, 0:1] + 1e-9)
        Zc, Zt = lg(e['Pcal']), lg(e['Pte'])
    r_cal, r_te, _ = density_ratio(Zc, Zt, e['seed'], CFG['W_CLIP'])
    _, wc, wt = temper(r_cal, r_te, CFG['ESS_MIN'])
    return wc, wt

ALPHAS = np.round(np.arange(0.02, 0.255, 0.01), 3)
VARIANTS = {'mondrian': None, 'wm_cov_pca': 'pca', 'wm_cov_score': 'score'}
rows_f = []
t0f = time.time()
for e in CACHE:
    ycal, yte, Pcal, Pte = e['ycal'], e['yte'], e['Pcal'], e['Pte']
    st = lac(Pcal)[np.arange(len(ycal)), ycal]; Ste = lac(Pte); nT = len(yte)
    W = {v: (weights_for(e, d) if d else None) for v, d in VARIANTS.items()}
    for a in ALPHAS:
        for v, wpair in W.items():
            T_ = np.zeros((nT, 2))
            for k in (0, 1):
                m = ycal == k
                T_[:, k] = threshold(st[m], a, nT) if wpair is None else threshold(st[m], a, nT, wpair[0][m], wpair[1])
            ev = evaluate(sets_from(Ste, T_), yte)
            rows_f.append(dict(site=e['site'], rep=e['rep'], alpha=a, variant=v, coverage=ev['coverage'], set_size=ev['set_size']))
F = pd.DataFrame(rows_f).groupby(['variant', 'alpha', 'site']).mean(numeric_only=True).reset_index()
tgt90 = 1 - CFG['ALPHA']
FR = F.groupby(['variant', 'alpha']).agg(set_size=('set_size', 'mean'), worst_site=('coverage', 'min'),
        p10_site=('coverage', lambda c: np.quantile(c, 0.10)),
        shortfall=('coverage', lambda c: np.clip(tgt90 - c, 0, None).mean()),
        n_below_085=('coverage', lambda c: int((c < 0.85).sum()))).reset_index()
FR.to_csv(f"{CFG['OUT_DIR']}/efficiency_frontier.csv", index=False)
print(f'frontier computed in {time.time()-t0f:.0f}s')

# Interpolate Mondrian's curve at each variant's set size: positive gap = the variant beats Mondrian at equal set size
mond = FR[FR.variant == 'mondrian'].sort_values('set_size')
cmp_rows = []
for v in ('wm_cov_pca', 'wm_cov_score'):
    for _, r in FR[FR.variant == v].iterrows():
        if mond.set_size.min() <= r.set_size <= mond.set_size.max():
            cmp_rows.append(dict(variant=v, alpha=r.alpha, set_size=r.set_size,
                gain_worst_site=r.worst_site - np.interp(r.set_size, mond.set_size, mond.worst_site),
                gain_p10_site=r.p10_site - np.interp(r.set_size, mond.set_size, mond.p10_site),
                shortfall_reduction=np.interp(r.set_size, mond.set_size, mond.shortfall) - r.shortfall))
CMP = pd.DataFrame(cmp_rows)
print('Gain over Mondrian at equal mean set size (positive = better than Mondrian), averaged over the alpha grid:')
display(CMP.groupby('variant')[['gain_worst_site', 'gain_p10_site', 'shortfall_reduction']].agg(['mean', 'min', 'max']).round(3))
print('Same, at the operating point alpha = 0.10:')
display(CMP[CMP.alpha == 0.10].round(3))
CMP.to_csv(f"{CFG['OUT_DIR']}/frontier_gain_vs_mondrian.csv", index=False)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
sty = {'mondrian': ('tab:orange', 's'), 'wm_cov_pca': ('tab:green', 'v'), 'wm_cov_score': ('tab:blue', 'o')}
for v, (c, mk) in sty.items():
    d = FR[FR.variant == v].sort_values('set_size')
    ax[0].plot(d.set_size, d.worst_site, marker=mk, c=c, ms=3, label=v)
    ax[1].plot(d.set_size, d.p10_site, marker=mk, c=c, ms=3, label=v)
    ax[2].plot(d.set_size, d.shortfall, marker=mk, c=c, ms=3, label=v)
for a_, t_ in zip(ax, ['worst-site coverage', '10th-percentile site coverage', 'mean shortfall below 0.90 (lower is better)']):
    a_.set_xlabel('mean set size'); a_.set_title(t_, fontsize=10)
ax[0].axhline(0.85, ls=':', c='r'); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{CFG['OUT_DIR']}/fig_frontier.png", dpi=200); plt.show()

In [ ]:
# ---------- 3g. Paper analyses: equal-set-size tests and the leave-site-out margin (replay, about 1-3 min) ----------
# Reproduces Table 2 (lower block and the margin row) and the equal-size statistics in Sect. 5.1 of the paper.
ALPHAS_G = np.round(np.arange(0.02, 0.2501, 0.0025), 4)
rows_g = []
for e in CACHE:
    yc, yt = e['ycal'], e['yte']; st = lac(e['Pcal'])[np.arange(len(yc)), yc]; Ste = lac(e['Pte']); n = len(yt)
    W = {'mondrian': None, 'pca': weights_for(e, 'pca'), 'score': weights_for(e, 'score')}
    for a in ALPHAS_G:
        for v, w in W.items():
            T_ = np.zeros((n, 2))
            for k in (0, 1):
                m = yc == k
                T_[:, k] = threshold(st[m], a, n) if w is None else threshold(st[m], a, n, w[0][m], w[1])
            S_ = Ste <= T_; sz = S_.sum(1); cov = S_[np.arange(n), yt]
            rows_g.append((e['site'], e['rep'], a, v, cov.mean(), sz.mean(), (sz == 1).mean(),
                           (S_.argmax(1) == yt)[sz == 1].mean() if (sz == 1).any() else np.nan))
G = pd.DataFrame(rows_g, columns=['site', 'rep', 'alpha', 'v', 'cov', 'size', 'single', 'sacc'])
G = G.groupby(['v', 'alpha', 'site']).mean(numeric_only=True).reset_index()
G.to_csv(f"{CFG['OUT_DIR']}/fine_grid_per_site.csv", index=False)
Mg = G[G.v == 'mondrian']
piv = {c: Mg.pivot(index='alpha', columns='site', values=c) for c in ['cov', 'size', 'single', 'sacc']}
A_ = piv['cov'].index.values
msz = piv['size'].mean(axis=1).values                       # mean set size decreases with alpha
def mondrian_at(alpha):
    return pd.DataFrame({c: pd.Series({s: np.interp(alpha, A_, piv[c][s].values) for s in piv[c].columns}) for c in piv})
def row_stats(name, d, ref=None):
    c = d['cov']; sf = np.clip(0.90 - c, 0, None)
    out = dict(method=name, mean_cov=c.mean(), worst=c.min(), below_085=int((c < 0.85).sum()), shortfall=sf.mean(),
               set_size=d['size'].mean(), singleton=d['single'].mean(), single_acc=d['sacc'].mean())
    if ref is not None:
        out['p_vs_mondrian'] = wilcoxon(sf, np.clip(0.90 - ref['cov'], 0, None), alternative='less', zero_method='zsplit').pvalue
    return out
m10 = mondrian_at(0.10)
TAB = [row_stats('Mondrian a=0.10', m10)]
rng_b = np.random.default_rng(0); EQ = []
for v in ('score', 'pca'):
    for a in (0.05, 0.075, 0.10, 0.125, 0.15, 0.20):
        g = G[(G.v == v) & np.isclose(G.alpha, a)].set_index('site')
        am = np.interp(g['size'].mean(), msz[::-1], A_[::-1]); mm = mondrian_at(am).loc[g.index]
        if np.isclose(a, 0.10):
            TAB.append(row_stats(f'WM-CP {v} a=0.10', g, m10))
            TAB.append(row_stats(f'Mondrian a={am:.3f} (matched to {v}, uses test labels)', mm))
        d = g['cov'] - mm['cov']
        try: p = wilcoxon(g['cov'], mm['cov'], alternative='greater').pvalue
        except ValueError: p = np.nan
        sites_ = g.index.values; bw = []
        for _ in range(4000):
            b = rng_b.choice(sites_, len(sites_), replace=True); bw.append((d[b]).mean())
        EQ.append(dict(variant=v, alpha=a, set_size=g['size'].mean(), matched_alpha=am, mean_cov_gain=d.mean(),
                       sites_better=int((d > 0).sum()), p_wilcoxon=p, ci_low=np.quantile(bw, .025), ci_high=np.quantile(bw, .975)))
# leave-site-out margin: per target site, pick alpha' from the OTHER sites' LOSO coverage curves (no target labels)
rules = {'shortfall<=0.005 (main)': lambda o: np.clip(0.90 - o, 0, None).mean(axis=1) <= 0.005,
         'no other site <0.85': lambda o: o.min(axis=1) >= 0.85,
         '10th pct site >=0.875': lambda o: o.quantile(0.10, axis=1) >= 0.875}
for rn, rule in rules.items():
    pick = {}
    for s_ in piv['cov'].columns:
        ok = rule(piv['cov'].drop(columns=s_)).values
        pick[s_] = A_[ok].max() if ok.any() else A_.min()
    d = pd.DataFrame({c: pd.Series({s_: piv[c].loc[pick[s_], s_] for s_ in pick}) for c in piv})
    r_ = row_stats(f'LSO margin, {rn}', d, m10); r_['alpha_range'] = f'{min(pick.values()):.4f}-{max(pick.values()):.4f}'; TAB.append(r_)
TAB = pd.DataFrame(TAB).set_index('method').round(4); EQ = pd.DataFrame(EQ).round(4)
print('Table 2 rows (alpha = 0.10, target 0.90):'); display(TAB)
print('Equal-set-size comparison against Mondrian (positive gain = method better):'); display(EQ)
TAB.to_csv(f"{CFG['OUT_DIR']}/paper_table2_extra.csv"); EQ.to_csv(f"{CFG['OUT_DIR']}/paper_equal_size_tests.csv", index=False)
g10 = G[(G.v == 'score') & np.isclose(G.alpha, 0.10)].set_index('site')
extra = g10['size'] - m10.loc[g10.index, 'size']; gap = 0.90 - m10.loc[g10.index, 'cov']
print('Adaptivity: Spearman(extra set size of score WM-CP, Mondrian coverage gap) =', spearmanr(extra, gap))

## 4. Results: coverage at unseen sites
*Worst-site coverage* and *# failing sites* are the numbers a clinician cares about: an average of 90% means little if one hospital gets 70%.

In [ ]:
# ---------- 4. Summary table ----------
METHODS = ['marginal', 'mondrian', 'wcp', 'wm_cov', 'wm_hyb']
site_mean = R.groupby(['method', 'site']).mean(numeric_only=True).reset_index()
target = 1 - ALPHA
def summarize(g):
    return pd.Series(dict(
        mean_cov=g.coverage.mean(), worst_site_cov=g.coverage.min(),
        n_fail_sites=int((g.coverage < target - CFG['FAIL_GAP']).sum()),
        mean_shortfall=np.clip(target - g.coverage, 0, None).mean(),
        worst_class_cov=np.nanmin(g[['cov_class0', 'cov_class1']].values.mean(0)),
        set_size=g.set_size.mean(), singleton=g.singleton.mean(), single_acc=g.single_acc.mean(),
        refer=g['refer'].mean(), empty_set=g['empty_set'].mean()))
SUMMARY = site_mean.groupby('method').apply(summarize).loc[METHODS].round(3)
SUMMARY.to_csv(f"{CFG['OUT_DIR']}/summary.csv")
print(f'Target coverage = {target:.2f}; {len(scored_sites)} scored sites x {CFG["N_REPEATS"]} calibration splits')
display(SUMMARY)
b = BASE.groupby('site').mean(numeric_only=True)
print(f"\nBase LOSO classifier: mean site accuracy {b.acc.mean():.3f}, balanced acc {b.bacc.mean():.3f}, AUC {b.auc.mean():.3f} "
      f"(site AUC range {b.auc.min():.2f}-{b.auc.max():.2f})")

In [ ]:
# ---------- 4a. Per-site uncertainty and paired tests ----------
from scipy.stats import beta, wilcoxon
def clopper_pearson(k, n, level=0.95):
    a = 1 - level
    lo = beta.ppf(a / 2, k, n - k + 1) if k > 0 else 0.0
    hi = beta.ppf(1 - a / 2, k + 1, n - k) if k < n else 1.0
    return lo, hi
r0 = R[R.rep == 0].copy()
ci = r0.apply(lambda r: clopper_pearson(int(round(r.coverage * r.n)), int(r.n)), axis=1)
r0['cov_lo'], r0['cov_hi'] = [c[0] for c in ci], [c[1] for c in ci]
CI = r0.pivot(index='site', columns='method', values='cov_lo')[METHODS].round(3)
print('Lower 95% Clopper-Pearson bound on coverage per site (calibration split 0):')
display(CI)
piv = site_mean.pivot(index='site', columns='method', values='coverage')
for m in ['wcp', 'wm_cov', 'wm_hyb']:
    sf_base = np.clip(target - piv['mondrian'], 0, None); sf_m = np.clip(target - piv[m], 0, None)
    try:
        p = wilcoxon(sf_m, sf_base, alternative='less', zero_method='zsplit').pvalue
    except ValueError:
        p = np.nan
    print(f'{m:7s} vs mondrian: mean shortfall {sf_m.mean():.3f} vs {sf_base.mean():.3f}, one-sided Wilcoxon p = {p:.3f}')
CI.to_csv(f"{CFG['OUT_DIR']}/coverage_lower_bounds.csv")

In [ ]:
# ---------- 4b. Figures ----------
order = site_mean[site_mean.method == 'marginal'].sort_values('coverage').site.tolist()
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw=dict(width_ratios=[3, 1]))
mk = dict(marginal='o', mondrian='s', wcp='^', wm_cov='v', wm_hyb='D')
for i, m in enumerate(METHODS):
    d = site_mean[site_mean.method == m].set_index('site').loc[order]
    ax[0].scatter(np.arange(len(order)) + (i - 2) * 0.12, d.coverage, marker=mk[m], s=28, label=m)
ax[0].axhline(target, ls='--', c='k', lw=1); ax[0].axhline(target - CFG['FAIL_GAP'], ls=':', c='r', lw=1)
ax[0].set_xticks(range(len(order))); ax[0].set_xticklabels(order, rotation=70, fontsize=8)
ax[0].set_ylabel('coverage at held-out site'); ax[0].legend(ncol=5, fontsize=8); ax[0].set_title('Per-site coverage (LOSO)')
ax[1].bar(METHODS, SUMMARY.set_size, color='0.6'); ax[1].set_ylabel('mean set size (max 2)')
ax[1].tick_params(axis='x', rotation=45); ax[1].set_title('Cost: set size')
plt.tight_layout(); plt.savefig(f"{CFG['OUT_DIR']}/fig_coverage.png", dpi=200); plt.show()

## 5. Site Readiness Score (label-free deployment gate)
For each new site, before any label exists, we estimate the coverage plain Mondrian CP will reach there
(importance-weighted calibration coverage with the hybrid weights). We then check it against the observed coverage.

In [ ]:
# ---------- 5. SRS validation ----------
S = SRS.groupby('site').mean(numeric_only=True)
obs = site_mean[site_mean.method == 'mondrian'].set_index('site').coverage
S['obs_gap'] = target - obs.loc[S.index]
rho_gap, p_gap = spearmanr(S.pred_gap, S.obs_gap)
rho_auc, p_auc = spearmanr(S.dom_auc, S.obs_gap)
rho_ess, p_ess = spearmanr(-S.ess_raw, S.obs_gap)
print(f'Spearman(predicted gap, observed gap) = {rho_gap:.2f} (p={p_gap:.3f})')
print(f'Spearman(domain AUC,   observed gap) = {rho_auc:.2f} (p={p_auc:.3f})')
print(f'Spearman(-ESS,         observed gap) = {rho_ess:.2f} (p={p_ess:.3f})')
print(f"Prevalence estimate error (EM, label-free): MAE = {np.abs(S.pi_tgt_est - S.pi_tgt_true).mean():.3f}")
flag = (S.pred_gap > CFG['GATE_PRED_GAP']) | (S.ess_raw < CFG['GATE_ESS'])
fail = S.obs_gap > CFG['FAIL_GAP']
print(f'\nGate (thresholds fixed in CFG, not tuned): flagged {flag.sum()}/{len(S)} sites; '
      f'caught {int((flag & fail).sum())}/{int(fail.sum())} failing sites; false alarms {int((flag & ~fail).sum())}')
S['flagged'] = flag; S['failed'] = fail
S.round(3).to_csv(f"{CFG['OUT_DIR']}/srs_by_site.csv")
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.scatter(S.pred_gap, S.obs_gap, c=np.where(fail, 'tab:red', 'tab:blue'))
for s_, r_ in S.iterrows(): ax.annotate(s_, (r_.pred_gap, r_.obs_gap), fontsize=7)
ax.axhline(CFG['FAIL_GAP'], ls=':', c='r'); ax.axvline(CFG['GATE_PRED_GAP'], ls=':', c='k')
ax.set_xlabel('predicted coverage loss (label-free)'); ax.set_ylabel('observed coverage loss (Mondrian)')
ax.set_title(f'Site Readiness Score, Spearman rho={rho_gap:.2f}')
plt.tight_layout(); plt.savefig(f"{CFG['OUT_DIR']}/fig_srs.png", dpi=200); plt.show()

## 6. Conformally gated biomarkers
For each held-out site we rank edges three ways: raw model weights, mean |attribution| over all subjects, and mean |attribution| over
subjects the shift-aware predictor classified with a singleton set. Stability = mean pairwise Kuncheva index of the top-K sets across sites.

In [ ]:
# ---------- 6. Biomarker stability ----------
def kuncheva(a, b, k, n):
    r = len(set(a) & set(b)); return (r * n - k * k) / (k * (n - k))
n_edges = len(TRIL[0]); K = min(CFG['TOPK'], n_edges // 4)
stab = {}
for variant in ('coef', 'all', 'conf'):
    tops = [np.argsort(-bio[s][variant])[:K] for s in bio]
    ki = [kuncheva(tops[i], tops[j], K, n_edges) for i in range(len(tops)) for j in range(i + 1, len(tops))]
    stab[variant] = (np.mean(ki), np.std(ki) / np.sqrt(len(ki)))
print('Top-%d edge stability across held-out sites (Kuncheva index, mean +/- s.e.):' % K)
for v, (m_, se) in stab.items(): print(f'  {v:5s}: {m_:.3f} +/- {se:.3f}')
print('Confident subjects per site (median):', int(np.median([bio[s]['n_conf'] for s in bio])))

freq = np.zeros(n_edges)
for s in bio: freq[np.argsort(-bio[s]['conf'])[:K]] += 1
top_edges = np.argsort(-freq)[:15]
BIOM = pd.DataFrame(dict(roi_a=[ROI_LABELS[TRIL[0][e]] for e in top_edges], roi_b=[ROI_LABELS[TRIL[1][e]] for e in top_edges],
                         selected_in_sites=freq[top_edges].astype(int), of_sites=len(bio)))
BIOM.to_csv(f"{CFG['OUT_DIR']}/stable_edges.csv", index=False)
display(BIOM)
plt.figure(figsize=(4.5, 3.5))
plt.bar(list(stab), [v[0] for v in stab.values()], yerr=[v[1] for v in stab.values()], color=['0.7', '0.5', 'tab:green'])
plt.ylabel('Kuncheva index'); plt.title('Cross-site biomarker stability'); plt.tight_layout()
plt.savefig(f"{CFG['OUT_DIR']}/fig_stability.png", dpi=200); plt.show()

## 7. Ablations to run for the paper (change CFG and re-run cells 3-6)
| Run | Change | What it tests |
|---|---|---|
| A1 | `CONN_KIND='correlation'` | Is the gain specific to tangent space? |
| A2 | `HARMONIZE='site_z'` | Does label-free harmonization replace shift-aware calibration, or remove signal at sites with skewed prevalence? |
| A3-A5 | domain space, ESS floor, alpha | Already computed in cell 3d from the cache, no rerun needed |
| A6 | `DERIVATIVE='rois_cc200'` | Atlas robustness (CC200 is the most used ABIDE atlas) |

Report every table as mean over sites with the worst site and the count of failing sites, never the pooled mean alone.

In [ ]:
# ---------- 8. Save everything ----------
with open(f"{CFG['OUT_DIR']}/run_config.json", 'w') as f: json.dump(CFG, f, indent=1)
print('Saved to', os.path.abspath(CFG['OUT_DIR']), ':', sorted(os.listdir(CFG['OUT_DIR'])))
if IN_COLAB:
    subprocess.run(['zip', '-qr', '/content/siteguard_results.zip', CFG['OUT_DIR']])
    from google.colab import files; files.download('/content/siteguard_results.zip')